# CLMBR Foundation Model

Uses the pretrained `StanfordShahLab/gpt-base-2048-clmbr` checkpoint to create fixed patient representations, then evaluates demographic + CLMBR and demographic + CLMBR + smoking models.

If `embeddings_final.parquet` is not already available, install the pinned CLMBR dependencies described in the repository README, request access to the gated Hugging Face checkpoint, and authenticate with a token that has been granted access.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

DATA_DIR = Path(
    os.environ.get("LUNG_CANCER_DATA_DIR", Path.home() / "lungcancer")
).expanduser()
RUN_DIR = DATA_DIR / "paper_reproduction"
PRED_DIR = RUN_DIR / "predictions"
FIG_DIR = RUN_DIR / "figures"
for directory in [RUN_DIR, PRED_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

COHORT_PATH = DATA_DIR / "demo_baseline_paper_reproduction.parquet"
if not COHORT_PATH.exists():
    raise FileNotFoundError(
        f"Missing {COHORT_PATH}. Run 03_PheCode_Model.ipynb first."
    )

demo = pd.read_parquet(COHORT_PATH).reset_index(drop=True)
cohort = demo.copy()
print("Using demo as the demographic + CLMBR modeling cohort")
cohort["person_id"] = pd.to_numeric(cohort["person_id"], errors="raise").astype("int64")
cohort["has_lungcancer_3Y"] = pd.to_numeric(
    cohort["has_lungcancer_3Y"], errors="raise"
).astype(int)

print("Cohort:", cohort.shape)
print("Cases:", int(cohort["has_lungcancer_3Y"].sum()))

In [ ]:
from sklearn.model_selection import StratifiedKFold

FOLD_PATH = RUN_DIR / "cv_folds.parquet"

def get_or_create_folds(df, n_splits=5, seed=42):
    expected = df[["person_id", "has_lungcancer_3Y"]].copy()
    if FOLD_PATH.exists():
        folds = pd.read_parquet(FOLD_PATH)
        check = expected.merge(folds, on="person_id", how="left", validate="one_to_one")
        if check["fold"].isna().any():
            raise ValueError("Saved fold file does not match the current cohort.")
        return check[["person_id", "fold"]]

    y = expected["has_lungcancer_3Y"]
    fold = np.full(len(expected), -1, dtype=int)
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fold_id, (_, test_idx) in enumerate(splitter.split(expected, y)):
        fold[test_idx] = fold_id
    folds = pd.DataFrame({"person_id": expected["person_id"], "fold": fold})
    folds.to_parquet(FOLD_PATH, index=False)
    return folds

folds = get_or_create_folds(cohort)
cohort = cohort.merge(folds, on="person_id", how="left", validate="one_to_one")
print(cohort.groupby("fold")["has_lungcancer_3Y"].agg(["count", "sum"]))

## Load or generate pre-index CLMBR embeddings

In [ ]:
EMBED_PATH = DATA_DIR / "embeddings_final.parquet"
MEDS_PATH = DATA_DIR / "EHR_MEDS.parquet"

if not EMBED_PATH.exists() and not MEDS_PATH.exists():
    raise FileNotFoundError(
        f"Need either {EMBED_PATH} or pre-index {MEDS_PATH}. "
        "The MEDS file must contain no events on or after each participant's index date."
    )

In [ ]:
if not EMBED_PATH.exists():
    import glob, time
    import polars as pl
    import torch
    from tqdm.auto import tqdm
    from transformers import AutoModelForCausalLM
    from hf_ehr.data.tokenization import CLMBRTokenizer
    from hf_ehr.config import Event

    MODEL_NAME = "StanfordShahLab/gpt-base-2048-clmbr"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device).eval()
    tokenizer = CLMBRTokenizer.from_pretrained(MODEL_NAME)
    ehr = pl.scan_parquet(MEDS_PATH)
    patient_ids = ehr.select(pl.col("subject_id").unique()).collect().to_series().to_list()

    def events_for(pid):
        rows = ehr.filter(pl.col("subject_id") == pid).sort("time").collect()
        rows = rows.tail(2000)
        return [Event(code="SNOMED/184099003" if r["code"] == "MEDS_BIRTH" else r["code"]) for r in rows.iter_rows(named=True)]

    ids, vectors = [], []
    for pid in tqdm(patient_ids, desc="CLMBR embeddings"):
        events = events_for(pid)
        if len(events) < 5:
            continue
        batch = tokenizer([events], add_special_tokens=True, padding=True, return_tensors="pt")
        input_ids = batch["input_ids"].to(device)
        mask = batch.get("attention_mask")
        mask = mask.to(device) if mask is not None else None
        with torch.no_grad():
            out = model(input_ids=input_ids, attention_mask=mask, output_hidden_states=True)
        pos = int(mask[0].sum().item() - 1) if mask is not None else -1
        vectors.append(out.hidden_states[-1][0, pos].detach().cpu().numpy())
        ids.append(pid)

    embeddings = pd.DataFrame(np.vstack(vectors), index=ids)
    embeddings.index.name = "person_id"
    embeddings.to_parquet(EMBED_PATH)
else:
    embeddings = pd.read_parquet(EMBED_PATH)

if "person_id" not in embeddings.columns:
    embeddings = embeddings.reset_index()
embeddings["person_id"] = pd.to_numeric(embeddings["person_id"], errors="raise").astype("int64")
print("Embeddings:", embeddings.shape)

## Shared-fold demographic + CLMBR models

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from xgboost import XGBClassifier

embedding_cols = [column for column in embeddings.columns if column != "person_id"]
data = cohort.merge(embeddings, on="person_id", how="left", validate="one_to_one")

embed = data[embedding_cols].apply(pd.to_numeric, errors="coerce")
embed["embedding_missing"] = embed.isna().all(axis=1).astype(int)

demographic_columns = [
    column for column in [
        "age_at_survey_int",
        "sex_at_birth",
        "race_ethnicity",
        "education_group",
    ]
    if column in data.columns
]
if "age_at_survey_int" not in demographic_columns:
    raise RuntimeError("age_at_survey_int is required for the demographic model")

demographic = data[demographic_columns].copy()
demographic["age_at_survey_int"] = pd.to_numeric(
    demographic["age_at_survey_int"], errors="coerce"
)
categorical_demographics = [
    column for column in demographic_columns if column != "age_at_survey_int"
]
demographic = pd.get_dummies(
    demographic,
    columns=categorical_demographics,
    dummy_na=True,
)
demographic = demographic.apply(pd.to_numeric, errors="coerce")

X = pd.concat([demographic, embed], axis=1)
X.columns = X.columns.astype(str)
X = X.loc[:, ~X.columns.duplicated()]
y = data["has_lungcancer_3Y"].astype(int)

print("Demographic source features:", demographic_columns)
print("Encoded demographic features:", demographic.shape[1])
print("CLMBR embedding features:", len(embedding_cols))
print("Smoking/lifestyle predictors included: 0")
print("Participants missing embeddings:", int(embed["embedding_missing"].sum()))

oof = np.zeros(len(data), dtype=float)
fold_aurocs = []
fold_auprcs = []

for fold_id in sorted(data["fold"].unique()):
    test = data["fold"].eq(fold_id).to_numpy()
    train = ~test

    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X.loc[train])
    X_test = imputer.transform(X.loc[test])

    model = XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )
    model.fit(X_train, y.loc[train])
    oof[test] = model.predict_proba(X_test)[:, 1]

    fold_auc = roc_auc_score(y.loc[test], oof[test])
    fold_ap = average_precision_score(y.loc[test], oof[test])
    fold_aurocs.append(fold_auc)
    fold_auprcs.append(fold_ap)
    print(f"Fold {fold_id + 1}: AUROC={fold_auc:.4f}; AUPRC={fold_ap:.4f}")

pred = pd.DataFrame({
    "person_id": data["person_id"],
    "y_true": y,
    "prob": oof,
    "model": "Demographic_CLMBR",
})
pred.to_parquet(PRED_DIR / "Demographic_CLMBR_predictions.parquet", index=False)

for stale_name in ["CLMBR_only_predictions.parquet", "Clinical_CLMBR_predictions.parquet"]:
    stale_path = PRED_DIR / stale_name
    if stale_path.exists():
        stale_path.unlink()
        print("Removed superseded prediction file:", stale_path)

print("OOF AUROC:", roc_auc_score(y, oof))
print("OOF AUPRC:", average_precision_score(y, oof))
print("OOF Brier:", brier_score_loss(y, oof))
print(f"Fold AUROC: {np.mean(fold_aurocs):.3f} ± {np.std(fold_aurocs):.3f}")
print(f"Fold AUPRC: {np.mean(fold_auprcs):.3f} ± {np.std(fold_auprcs):.3f}")


## Demographic + CLMBR + smoking model

Adds smoking status, smoking duration, usual cigarettes per day, derived pack-years, and years since quitting to the demographic + CLMBR feature set. It uses the same participants, folds, preprocessing, and XGBoost settings as the demographic + CLMBR model.


In [ ]:
def numeric_data_column(frame, name):
    if name not in frame.columns:
        return pd.Series(np.nan, index=frame.index, dtype=float, name=name)
    return pd.to_numeric(frame[name], errors="coerce")

if "smoking_status" not in data.columns:
    raise RuntimeError("smoking_status is required for the smoking-augmented model")

smoking_status = (
    data["smoking_status"].astype("string").str.strip().str.lower()
)
smoking_years = numeric_data_column(data, "Smoking: Number Of Years")
average_daily = numeric_data_column(
    data, "Smoking: Average Daily Cigarette Number"
)
current_daily = numeric_data_column(
    data, "Smoking: Current Daily Cigarette Number"
)
quit_age = numeric_data_column(
    data, "Attempt Quit Smoking: Completely Quit Age"
)
age_numeric = numeric_data_column(data, "age_at_survey_int")

# Prefer usual/lifetime cigarettes per day. Current daily cigarettes are a
# fallback only for current smokers; zero is treated as missing intensity.
cigs_per_day = average_daily.mask(average_daily <= 0)
current_fallback = smoking_status.eq("current") & cigs_per_day.isna()
valid_current_daily = current_daily.mask(current_daily <= 0)
cigs_per_day.loc[current_fallback] = valid_current_daily.loc[current_fallback]

years_since_quit = pd.Series(np.nan, index=data.index, dtype=float)
years_since_quit.loc[smoking_status.eq("current")] = 0.0
former = smoking_status.eq("former")
years_since_quit.loc[former] = age_numeric.loc[former] - quit_age.loc[former]
years_since_quit = years_since_quit.mask(years_since_quit < 0)

smoking = pd.DataFrame({
    "smoking_status": smoking_status,
    "smoking_years": smoking_years,
    "cigs_per_day": cigs_per_day,
    "pack_years": (cigs_per_day / 20.0) * smoking_years,
    "years_since_quit": years_since_quit,
})
smoking = pd.get_dummies(
    smoking,
    columns=["smoking_status"],
    dummy_na=True,
)
smoking = smoking.apply(pd.to_numeric, errors="coerce")

X_smoking = pd.concat([demographic, smoking, embed], axis=1)
X_smoking.columns = X_smoking.columns.astype(str)
X_smoking = X_smoking.loc[:, ~X_smoking.columns.duplicated()]

print("Smoking features:", list(smoking.columns))
print("Missing usable cigarettes/day:", int(cigs_per_day.isna().sum()))
print("Missing smoking duration:", int(smoking_years.isna().sum()))
print("Former smokers missing quit age:", int((former & quit_age.isna()).sum()))

oof_smoking = np.zeros(len(data), dtype=float)
smoking_fold_aurocs = []
smoking_fold_auprcs = []

for fold_id in sorted(data["fold"].unique()):
    test = data["fold"].eq(fold_id).to_numpy()
    train = ~test

    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_smoking.loc[train])
    X_test = imputer.transform(X_smoking.loc[test])

    smoking_model = XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )
    smoking_model.fit(X_train, y.loc[train])
    oof_smoking[test] = smoking_model.predict_proba(X_test)[:, 1]

    fold_auc = roc_auc_score(y.loc[test], oof_smoking[test])
    fold_ap = average_precision_score(y.loc[test], oof_smoking[test])
    smoking_fold_aurocs.append(fold_auc)
    smoking_fold_auprcs.append(fold_ap)
    print(f"Fold {fold_id + 1}: AUROC={fold_auc:.4f}; AUPRC={fold_ap:.4f}")

pred_smoking = pd.DataFrame({
    "person_id": data["person_id"],
    "y_true": y,
    "prob": oof_smoking,
    "model": "Demographic_CLMBR_Smoking",
})
pred_smoking.to_parquet(
    PRED_DIR / "Demographic_CLMBR_Smoking_predictions.parquet",
    index=False,
)

print("OOF AUROC:", roc_auc_score(y, oof_smoking))
print("OOF AUPRC:", average_precision_score(y, oof_smoking))
print("OOF Brier:", brier_score_loss(y, oof_smoking))
print(
    f"Fold AUROC: {np.mean(smoking_fold_aurocs):.3f} ± "
    f"{np.std(smoking_fold_aurocs):.3f}"
)
print(
    f"Fold AUPRC: {np.mean(smoking_fold_auprcs):.3f} ± "
    f"{np.std(smoking_fold_auprcs):.3f}"
)


## Clinician-facing fixed-capacity performance

Participants are ranked by out-of-fold predicted risk. Sensitivity and PPV are reported after selecting the top 5%, 10%, and 20% of the cohort.


In [ ]:
def top_capacity_metrics(predictions, fractions=(0.05, 0.10, 0.20)):
    ranked = predictions.sort_values(
        ["prob", "person_id"], ascending=[False, True]
    ).reset_index(drop=True)
    y_values = ranked["y_true"].astype(int).to_numpy()
    rows = []

    for fraction in fractions:
        selected_n = int(np.floor(len(ranked) * fraction))
        selected = np.zeros(len(ranked), dtype=bool)
        selected[:selected_n] = True

        tp = int((selected & (y_values == 1)).sum())
        fp = int((selected & (y_values == 0)).sum())
        fn = int((~selected & (y_values == 1)).sum())
        tn = int((~selected & (y_values == 0)).sum())

        rows.append({
            "Capacity": f"Top {int(fraction * 100)}%",
            "Selected": selected_n,
            "Cases captured": tp,
            "Sensitivity": tp / (tp + fn),
            "Specificity": tn / (tn + fp),
            "PPV": tp / (tp + fp),
            "Screened per observed case": selected_n / tp if tp else np.inf,
        })

    return pd.DataFrame(rows).set_index("Capacity")


clmbr_capacity_metrics = top_capacity_metrics(pred)
clmbr_smoking_capacity_metrics = top_capacity_metrics(pred_smoking)
capacity_comparison = pd.concat(
    {
        "Demographic + CLMBR": clmbr_capacity_metrics,
        "Demographic + CLMBR + Smoking": clmbr_smoking_capacity_metrics,
    },
    names=["Model", "Capacity"],
)
display(capacity_comparison.style.format({
    "Sensitivity": "{:.1%}",
    "Specificity": "{:.1%}",
    "PPV": "{:.1%}",
    "Screened per observed case": "{:.1f}",
}))

def slide_summary_row(label, predictions, capacity_metrics):
    row = {
        "Model": label,
        "AUROC": roc_auc_score(predictions["y_true"], predictions["prob"]),
        "AUPRC": average_precision_score(
            predictions["y_true"], predictions["prob"]
        ),
    }
    for fraction in (5, 10, 20):
        capacity = f"Top {fraction}%"
        row[f"Top {fraction}% Sens./PPV"] = (
            f"{capacity_metrics.loc[capacity, 'Sensitivity']:.1%} / "
            f"{capacity_metrics.loc[capacity, 'PPV']:.1%}"
        )
    return row

clmbr_summary = pd.DataFrame([
    slide_summary_row(
        "Demographic + CLMBR", pred, clmbr_capacity_metrics
    ),
    slide_summary_row(
        "Demographic + CLMBR + Smoking",
        pred_smoking,
        clmbr_smoking_capacity_metrics,
    ),
]).set_index("Model")

display(clmbr_summary.style.format({"AUROC": "{:.3f}", "AUPRC": "{:.3f}"}))
clmbr_summary.to_csv(RUN_DIR / "CLMBR_slide_metrics.csv")
clmbr_summary.loc[["Demographic + CLMBR"]].to_csv(
    RUN_DIR / "Demographic_CLMBR_slide_metrics.csv"
)
clmbr_summary.loc[["Demographic + CLMBR + Smoking"]].to_csv(
    RUN_DIR / "Demographic_CLMBR_Smoking_slide_metrics.csv"
)
